<a href="https://colab.research.google.com/github/Ernest-Trevis/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Practical Exercise 4: Developing a Knowledge Based Agent**

Setup to run first:


In [ ]:
!mkdir -p trailer data tests

In [ ]:
%%writefile trailer/__init__.py
"""Self-driving trailer: a knowledge-based agent for freight routing in Kenya."""

Writing trailer/__init__.py


**Knowledge Base:**

In [ ]:
%%writefile data/network.json
{
  "_note": "Road distances (km) are approximate. Replace with your Practical Exercise 3 graph, or refresh via trailer/api_loader.py.",
  "places": [
    {"name": "Mombasa",     "lat": -4.0435, "lon": 39.6682, "fuel": true,  "rest": true},
    {"name": "Voi",         "lat": -3.3963, "lon": 38.5560, "fuel": true,  "rest": true},
    {"name": "Mtito Andei", "lat": -2.6906, "lon": 38.1646, "fuel": true,  "rest": true},
    {"name": "Nairobi",     "lat": -1.2921, "lon": 36.8219, "fuel": true,  "rest": true},
    {"name": "Thika",       "lat": -1.0332, "lon": 37.0693, "fuel": true,  "rest": true},
    {"name": "Nyeri",       "lat": -0.4167, "lon": 36.9500, "fuel": true,  "rest": true},
    {"name": "Naivasha",    "lat": -0.7172, "lon": 36.4310, "fuel": false, "rest": false},
    {"name": "Nakuru",      "lat": -0.3031, "lon": 36.0800, "fuel": true,  "rest": true},
    {"name": "Kericho",     "lat": -0.3689, "lon": 35.2863, "fuel": false, "rest": true},
    {"name": "Eldoret",     "lat":  0.5143, "lon": 35.2698, "fuel": true,  "rest": true},
    {"name": "Kisumu",      "lat": -0.0917, "lon": 34.7680, "fuel": true,  "rest": true},
    {"name": "Malaba",      "lat":  0.6360, "lon": 34.2740, "fuel": true,  "rest": true}
  ],
  "roads": [
    {"from": "Mombasa",     "to": "Voi",         "km": 160, "speed_kmh": 80, "road": "A109"},
    {"from": "Voi",         "to": "Mtito Andei", "km": 110, "speed_kmh": 80, "road": "A109"},
    {"from": "Mtito Andei", "to": "Nairobi",     "km": 235, "speed_kmh": 80, "road": "A109"},
    {"from": "Nairobi",     "to": "Thika",       "km": 45,  "speed_kmh": 80, "road": "A2"},
    {"from": "Thika",       "to": "Nyeri",       "km": 100, "speed_kmh": 80, "road": "A2"},
    {"from": "Nyeri",       "to": "Nakuru",      "km": 160, "speed_kmh": 70, "road": "B5", "max_tonnes": 30},
    {"from": "Nairobi",     "to": "Naivasha",    "km": 90,  "speed_kmh": 80, "road": "A104"},
    {"from": "Naivasha",    "to": "Nakuru",      "km": 65,  "speed_kmh": 80, "road": "A104"},
    {"from": "Nakuru",      "to": "Kericho",     "km": 100, "speed_kmh": 70, "road": "A104"},
    {"from": "Kericho",     "to": "Kisumu",      "km": 75,  "speed_kmh": 70, "road": "A1", "max_tonnes": 25},
    {"from": "Nakuru",      "to": "Eldoret",     "km": 155, "speed_kmh": 80, "road": "A104"},
    {"from": "Eldoret",     "to": "Kisumu",      "km": 115, "speed_kmh": 70, "road": "B1"},
    {"from": "Eldoret",     "to": "Malaba",      "km": 120, "speed_kmh": 80, "road": "A104"}
  ]
}

Writing data/network.json


In [ ]:
%%writefile trailer/kb.py

from __future__ import annotations

import json
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Optional

DEFAULT_DATA = Path(__file__).resolve().parent.parent / "data" / "network.json"


@dataclass(frozen=True)
class Place:
    name: str
    lat: float
    lon: float
    fuel: bool = False
    rest: bool = False


@dataclass(frozen=True)
class Edge:
    src: str
    dst: str
    km: float
    speed_kmh: int = 80
    road: str = ""
    max_tonnes: Optional[float] = None

    @property
    def hours(self) -> float:
        return self.km / self.speed_kmh


def _norm(x) -> str:
    return str(x).strip().lower().replace("_", " ")


class KnowledgeBase:
    def __init__(self) -> None:
        self.places: dict[str, Place] = {}
        self._out: dict[str, dict[str, Edge]] = {}
        self._dynamic: set[tuple] = set()

    def add_place(self, place: Place) -> None:
        self.places[place.name] = place
        self._out.setdefault(place.name, {})

    def add_edge(self, edge: Edge, oneway: bool = False) -> None:
        for n in (edge.src, edge.dst):
            if n not in self.places:
                raise KeyError(f"Unknown place: {n}")
        self._out[edge.src][edge.dst] = edge
        if not oneway:
            self._out[edge.dst][edge.src] = replace(edge, src=edge.dst, dst=edge.src)

    @classmethod
    def from_json(cls, path=None) -> "KnowledgeBase":
        data = json.loads(Path(path or DEFAULT_DATA).read_text(encoding="utf-8"))
        kb = cls()
        for p in data["places"]:
            kb.add_place(Place(p["name"], p["lat"], p["lon"], p.get("fuel", False), p.get("rest", False)))
        for r in data["roads"]:
            kb.add_edge(
                Edge(r["from"], r["to"], r["km"], r.get("speed_kmh", 80), r.get("road", ""), r.get("max_tonnes")),
                oneway=r.get("oneway", False),
            )
        return kb

    def resolve(self, name: str) -> str:
        for p in self.places:
            if _norm(p) == _norm(name):
                return p
        raise KeyError(f"Unknown place: {name!r}. Known: {', '.join(sorted(self.places))}")

    def neighbors(self, name: str) -> list[Edge]:
        return list(self._out.get(name, {}).values())

    def edge(self, a: str, b: str) -> Optional[Edge]:
        return self._out.get(a, {}).get(b)

    def edges(self) -> list[Edge]:
        return [e for out in self._out.values() for e in out.values()]

    def as_facts(self) -> set[tuple]:
        facts: set[tuple] = set()
        for p in self.places.values():
            facts.add(("place", p.name))
            facts.add(("at", p.name, p.lat, p.lon))
            if p.fuel:
                facts.add(("fuel_station", p.name))
            if p.rest:
                facts.add(("rest_stop", p.name))
        for e in self.edges():
            facts.add(("road", e.src, e.dst, e.km))
            facts.add(("speed_limit", e.src, e.dst, e.speed_kmh))
            if e.max_tonnes is not None:
                facts.add(("max_tonnes", e.src, e.dst, e.max_tonnes))
        return facts | self._dynamic

    def tell(self, *fact) -> None:
        self._dynamic.add(tuple(fact))

    def retract(self, *fact) -> None:
        self._dynamic.discard(tuple(fact))

    def ask(self, *pattern) -> list[tuple]:
        out = []
        for f in sorted(self.as_facts(), key=str):
            if len(f) != len(pattern):
                continue
            if all(p is None or p == "*" or _norm(p) == _norm(v) for p, v in zip(pattern, f)):
                out.append(f)
        return out

    def close(self, a: str, b: str, both: bool = True) -> None:
        self.tell("closed", a, b)
        if both:
            self.tell("closed", b, a)

    def reopen(self, a: str, b: str, both: bool = True) -> None:
        self.retract("closed", a, b)
        if both:
            self.retract("closed", b, a)

Writing trailer/kb.py


In [ ]:
%%writefile trailer/distances.py

from __future__ import annotations

import math

EARTH_R = 6371.0088


def _xy(a, b):
    mean_lat = math.radians((a.lat + b.lat) / 2)
    dx = EARTH_R * math.radians(b.lon - a.lon) * math.cos(mean_lat)
    dy = EARTH_R * math.radians(b.lat - a.lat)
    return dx, dy


def haversine_km(a, b) -> float:
    p1, p2 = math.radians(a.lat), math.radians(b.lat)
    dphi, dl = p2 - p1, math.radians(b.lon - a.lon)
    h = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * EARTH_R * math.asin(math.sqrt(h))


def euclidean_km(a, b) -> float:
    return math.hypot(*_xy(a, b))


def manhattan_km(a, b) -> float:
    dx, dy = _xy(a, b)
    return abs(dx) + abs(dy)


def road_km(kb, path) -> float:
    return sum(kb.edge(a, b).km for a, b in zip(path, path[1:]))


def compare(kb, a: str, b: str, path=None) -> dict:
    pa, pb = kb.places[a], kb.places[b]
    out = {
        "haversine_km": round(haversine_km(pa, pb), 1),
        "euclidean_km": round(euclidean_km(pa, pb), 1),
        "manhattan_km": round(manhattan_km(pa, pb), 1),
    }
    if path:
        out["road_km"] = round(road_km(kb, path), 1)
    return out

Writing trailer/distances.py


In [ ]:
%%writefile trailer/api_loader.py

from __future__ import annotations

import json
import time
import urllib.request
from dataclasses import replace

OSRM = "https://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=false"


def fetch_road(a, b, timeout: int = 10):
    url = OSRM.format(lon1=a.lon, lat1=a.lat, lon2=b.lon, lat2=b.lat)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "strathmore-trailer-kba/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            route = json.load(r)["routes"][0]
        return route["distance"] / 1000, route["duration"] / 3600
    except Exception as exc:  # network down, rate limited, bad JSON
        print(f"[api_loader] {a.name}->{b.name} failed: {exc}")
        return None


def refresh_distances(kb, pause: float = 1.0) -> int:
    updated = 0
    for e in kb.edges():
        got = fetch_road(kb.places[e.src], kb.places[e.dst])
        if got:
            kb._out[e.src][e.dst] = replace(e, km=round(got[0], 1))
            updated += 1
        time.sleep(pause)
    return updated


if __name__ == "__main__":
    from .kb import KnowledgeBase

    kb = KnowledgeBase.from_json()
    print("updated", refresh_distances(kb), "of", len(kb.edges()), "directed edges")


Writing trailer/api_loader.py


In [ ]:
from trailer.kb import KnowledgeBase
from trailer.distances import compare
kb = KnowledgeBase.from_json()
print(len(kb.places), 'places,', len(kb.edges()), 'directed edges')
print(kb.ask('fuel_station', '*'))
print(compare(kb, 'Nairobi', 'Mombasa'))

12 places, 26 directed edges
[('fuel_station', 'Eldoret'), ('fuel_station', 'Kisumu'), ('fuel_station', 'Malaba'), ('fuel_station', 'Mombasa'), ('fuel_station', 'Mtito Andei'), ('fuel_station', 'Nairobi'), ('fuel_station', 'Nakuru'), ('fuel_station', 'Nyeri'), ('fuel_station', 'Thika'), ('fuel_station', 'Voi')]
{'haversine_km': 439.9, 'euclidean_km': 439.9, 'manhattan_km': 622.1}


# Reasoning
Implemeted using BFS, Greedy algorithm, forward chaining rules, fuel and rest planning.

In [ ]:
%%writefile trailer/search.py

from __future__ import annotations

import heapq
import itertools
import sys
from dataclasses import dataclass
from typing import Optional

from .distances import haversine_km

STRATEGIES = ("bfs", "ucs", "greedy", "astar")


@dataclass
class Node:
    id: int
    state: str
    parent: Optional["Node"]
    g: float       # road km from start
    h: float       # heuristic km to goal (haversine) - 0 for uninformed strategies
    depth: int

    @property
    def f(self) -> float:
        return self.g + self.h


@dataclass
class SearchResult:
    strategy: str
    start: str
    goal: str
    path: Optional[list]
    cost_km: float
    expanded: list      # nodes in expansion order
    generated: list     # every node put on the frontier (the search tree)

    @property
    def found(self) -> bool:
        return self.path is not None


def search(kb, start: str, goal: str, strategy: str = "astar", forbidden=frozenset()) -> SearchResult:
    if strategy not in STRATEGIES:
        raise ValueError(f"strategy must be one of {STRATEGIES}")
    informed = strategy in ("greedy", "astar")
    h = (lambda s: haversine_km(kb.places[s], kb.places[goal])) if informed else (lambda s: 0.0)
    prio = {
        "bfs": lambda n: n.depth,
        "ucs": lambda n: n.g,
        "greedy": lambda n: n.h,
        "astar": lambda n: n.f,
    }[strategy]

    ids = itertools.count()
    root = Node(next(ids), start, None, 0.0, h(start), 0)
    generated, expanded = [root], []
    frontier = [(prio(root), root.id, root)]
    best_g = {start: 0.0}
    closed = set()

    while frontier:
        _, _, node = heapq.heappop(frontier)
        if node.state in closed:
            continue
        closed.add(node.state)
        expanded.append(node)
        if node.state == goal:
            path = []
            n = node
            while n:
                path.append(n.state)
                n = n.parent
            return SearchResult(strategy, start, goal, path[::-1], node.g, expanded, generated)
        for e in kb.neighbors(node.state):
            if (e.src, e.dst) in forbidden or e.dst in closed:
                continue
            g = node.g + e.km
            if strategy in ("ucs", "astar"):
                if g >= best_g.get(e.dst, float("inf")):
                    continue
            elif e.dst in best_g:
                continue
            best_g[e.dst] = g
            child = Node(next(ids), e.dst, node, g, h(e.dst), node.depth + 1)
            generated.append(child)
            heapq.heappush(frontier, (prio(child), child.id, child))
    return SearchResult(strategy, start, goal, None, float("inf"), expanded, generated)


def format_tree(result: SearchResult) -> str:
    """Text rendering of the search tree: [#k] = k-th expansion, [frontier] = generated, never expanded."""
    order = {n.id: i + 1 for i, n in enumerate(result.expanded)}
    kids: dict = {}
    for n in result.generated:
        if n.parent:
            kids.setdefault(n.parent.id, []).append(n)
    on_path = set(result.path or [])
    lines = [f"Search tree ({result.strategy}) {result.start} -> {result.goal}"]

    def walk(n: Node, indent: str) -> None:
        tag = f"[#{order[n.id]}]" if n.id in order else "[frontier]"
        star = " *" if n.state in on_path and n.id in order else ""
        lines.append(f"{indent}{n.state} {tag} g={n.g:.0f} h={n.h:.0f} f={n.f:.0f}{star}")
        for c in kids.get(n.id, []):
            walk(c, indent + "    ")

    walk(result.generated[0], "")
    lines.append(f"expanded={len(result.expanded)} generated={len(result.generated)} "
                 f"cost={result.cost_km:.0f} km path={' > '.join(result.path) if result.path else 'NONE'}")
    return "\n".join(lines)


def print_tree(result: SearchResult, file=None) -> None:
    print(format_tree(result), file=file or sys.stdout)


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

@dataclass
class TrailerSpec:
    gross_tonnes: float = 28.0   # loaded weight in trailer
    tank_l: float = 300.0
    km_per_l: float = 2.5
    reserve_km: float = 50.0     # never plan to arrive on fumes
    max_drive_h: float = 4.5     # continuous driving before a break (company policy, configurable)
    rest_min: int = 45

    @property
    def range_km(self) -> float:
        return self.tank_l * self.km_per_l


# Implemeting forward chaining
def r_overweight(facts, t):
    return {("overweight", f[1], f[2], f"{t.gross_tonnes:g} t trailer exceeds {f[3]:g} t limit")
            for f in facts if f[0] == "max_tonnes" and t.gross_tonnes > f[3]}


def r_closed(facts, t):
    return {("blocked", f[1], f[2], "road closed") for f in facts if f[0] == "closed"}


def r_blocked_from_overweight(facts, t):
    return {("blocked", f[1], f[2], f[3]) for f in facts if f[0] == "overweight"}


def r_forbidden(facts, t):
    return {("forbidden", f[1], f[2]) for f in facts if f[0] == "blocked"}


RULES = (r_overweight, r_closed, r_blocked_from_overweight, r_forbidden)


@dataclass
class Inference:
    facts: set
    forbidden: set = field(default_factory=set)
    reasons: dict = field(default_factory=dict)   # (a, b) -> [reason, ...]
    rounds: int = 0


def infer(kb, trailer: TrailerSpec) -> Inference:
    facts = set(kb.as_facts())
    rounds = 0
    while True:
        new = set()
        for rule in RULES:
            new |= rule(facts, trailer)
        new -= facts
        if not new:
            break
        facts |= new
        rounds += 1
    inf = Inference(facts, rounds=rounds)
    for f in facts:
        if f[0] == "forbidden":
            inf.forbidden.add((f[1], f[2]))
        elif f[0] == "blocked":
            inf.reasons.setdefault((f[1], f[2]), []).append(f[3])
    return inf


# ---------------- trip planning ----------------
@dataclass
class TripPlan:
    events: list          # ("drive",a,b,km,h) | ("refuel",node,litres) | ("rest",node,minutes)
    feasible: bool
    km: float
    drive_h: float
    total_h: float

    def format(self) -> str:
        out = []
        for ev in self.events:
            if ev[0] == "drive":
                out.append(f"  drive   {ev[1]} -> {ev[2]}  {ev[3]:.0f} km, {ev[4]:.1f} h")
            elif ev[0] == "refuel":
                out.append(f"  refuel  at {ev[1]}  (+{ev[2]:.0f} L)")
            else:
                out.append(f"  rest    at {ev[1]}  ({ev[2]} min)")
        out.append(f"  total: {self.km:.0f} km, {self.drive_h:.1f} h driving, {self.total_h:.1f} h incl. stops"
                   + ("" if self.feasible else "   ** NOT FEASIBLE with current fuel/rest facilities **"))
        return "\n".join(out)


def _stops(costs, capacity, used0, available):
    """Look-ahead rule: at a facility, stop iff we could not reach the next facility (or the end) without it."""
    stops, used, ok, n = [], used0, True, len(costs)
    for i in range(n):
        need, k = 0.0, i
        while k < n:
            need += costs[k]
            k += 1
            if k < n and available[k]:
                break
        if available[i] and used > 0 and used + need > capacity + 1e-9:
            stops.append(i)
            used = 0.0
        if used + costs[i] > capacity + 1e-9:
            ok = False
        used += costs[i]
    return set(stops), ok


def plan_trip(kb, trailer: TrailerSpec, path, fuel_l=None, hours_since_rest: float = 0.0) -> TripPlan:
    fuel = trailer.tank_l if fuel_l is None else fuel_l
    edges = [kb.edge(a, b) for a, b in zip(path, path[1:])]
    fuel_stops, f_ok = _stops([e.km for e in edges], trailer.range_km - trailer.reserve_km,
                              (trailer.tank_l - fuel) * trailer.km_per_l, [kb.places[n].fuel for n in path])
    rest_stops, r_ok = _stops([e.hours for e in edges], trailer.max_drive_h,
                              hours_since_rest, [kb.places[n].rest for n in path])
    events, km, hrs, stop_h = [], 0.0, 0.0, 0.0
    for i, n in enumerate(path):
        if i in rest_stops:
            events.append(("rest", n, trailer.rest_min))
            stop_h += trailer.rest_min / 60
        if i in fuel_stops:
            events.append(("refuel", n, round(trailer.tank_l - fuel, 1)))
            fuel = trailer.tank_l
        if i < len(edges):
            e = edges[i]
            events.append(("drive", e.src, e.dst, e.km, e.hours))
            fuel -= e.km / trailer.km_per_l
            km += e.km
            hrs += e.hours
    return TripPlan(events, f_ok and r_ok, km, hrs, hrs + stop_h)


In [ ]:
from trailer.search import search, format_tree, STRATEGIES
from trailer.rules import TrailerSpec, infer, plan_trip
print(format_tree(search(kb, 'Nairobi', 'Kisumu', 'astar')))
for s in STRATEGIES:
    r = search(kb, 'Nairobi', 'Kisumu', s)
    print(f'{s:7} cost={r.cost_km:.0f} km expanded={len(r.expanded)}')
print(infer(kb, TrailerSpec(gross_tonnes=28)).reasons)
print(plan_trip(kb, TrailerSpec(), search(kb, 'Mombasa', 'Malaba').path).format())

# Agent and Interaction:

In [ ]:
%%writefile trailer/agent.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from .distances import compare
from .kb import KnowledgeBase
from .rules import Inference, TripPlan, TrailerSpec, infer, plan_trip
from .search import SearchResult, search


@dataclass
class Plan:
    start: str
    goal: str
    route: Optional[list]
    km: float
    trip: Optional[TripPlan]
    search: SearchResult
    inference: Inference
    explanation: str

    @property
    def ok(self) -> bool:
        return self.route is not None and self.trip is not None and self.trip.feasible


class TrailerAgent:
    def __init__(self, kb: KnowledgeBase, trailer: Optional[TrailerSpec] = None, position: str = "Nairobi",
                 fuel_l: Optional[float] = None, hours_since_rest: float = 0.0, strategy: str = "astar"):
        self.kb = kb
        self.trailer = trailer or TrailerSpec()
        self.position = kb.resolve(position)
        self.fuel_l = self.trailer.tank_l if fuel_l is None else fuel_l
        self.hours_since_rest = hours_since_rest
        self.strategy = strategy

    # ---- knowledge interface ----
    def tell(self, *fact):
        self.kb.tell(*fact)

    def ask(self, *pattern):
        return self.kb.ask(*pattern)

    def percept(self, kind: str, *args) -> str:
        """Sensor input -> KB update. kinds: close, open."""
        a, b = (self.kb.resolve(x) for x in args)
        if kind == "close":
            self.kb.close(a, b)
            return f"PERCEPT: road {a}-{b} is closed"
        if kind == "open":
            self.kb.reopen(a, b)
            return f"PERCEPT: road {a}-{b} reopened"
        raise ValueError(kind)

    # ---- deliberation ----
    def plan(self, goal: str, strategy: Optional[str] = None, start: Optional[str] = None) -> Plan:
        start = self.kb.resolve(start or self.position)
        goal = self.kb.resolve(goal)
        strategy = strategy or self.strategy
        inf = infer(self.kb, self.trailer)
        res = search(self.kb, start, goal, strategy, inf.forbidden)
        lines = [f"Strategy: {strategy}; trailer {self.trailer.gross_tonnes:g} t"]
        for (a, b), why in sorted(inf.reasons.items()):
            if self.kb.edge(a, b):
                lines.append(f"Avoided {a} -> {b}: {'; '.join(sorted(set(why)))}")
        if not res.found:
            lines.append(f"No legal route from {start} to {goal}.")
            return Plan(start, goal, None, float("inf"), None, res, inf, "\n".join(lines))
        trip = plan_trip(self.kb, self.trailer, res.path, self.fuel_l, self.hours_since_rest)
        lines.append(f"Route: {' > '.join(res.path)} ({res.cost_km:.0f} km, "
                     f"{len(res.expanded)} nodes expanded, {len(res.generated)} generated)")
        return Plan(start, goal, res.path, res.cost_km, trip, res, inf, "\n".join(lines))

    def distances(self, a: str, b: str) -> dict:
        a, b = self.kb.resolve(a), self.kb.resolve(b)
        res = search(self.kb, a, b, "astar", infer(self.kb, self.trailer).forbidden)
        return compare(self.kb, a, b, res.path)

    # ---- acting (simulation) ----
    def drive(self, goal: str, scripted_percepts: Optional[dict] = None, log=print) -> bool:
        """Follow the plan leg by leg. scripted_percepts = {arrival_place: [("close", "A", "B"), ...]}.
        If a percept makes the rest of the route illegal, re-plan from the current position."""
        scripted = {self.kb.resolve(k): list(v) for k, v in (scripted_percepts or {}).items()}
        goal = self.kb.resolve(goal)
        plan = self.plan(goal)
        log(plan.explanation)
        if not plan.ok:
            log("Cannot complete this trip." if plan.route is None else plan.trip.format())
            return False
        while self.position != goal:
            replanned = False
            for ev in plan.trip.events:
                if ev[0] == "rest":
                    self.hours_since_rest = 0.0
                    log(f"[{self.position}] rest {ev[2]} min")
                elif ev[0] == "refuel":
                    self.fuel_l = self.trailer.tank_l
                    log(f"[{self.position}] refuel +{ev[2]:.0f} L (tank full)")
                else:
                    _, a, b, km, h = ev
                    self.fuel_l -= km / self.trailer.km_per_l
                    self.hours_since_rest += h
                    self.position = b
                    log(f"[{a}] -> [{b}]  {km:.0f} km | fuel {self.fuel_l:.0f} L | {self.hours_since_rest:.1f} h since rest")
                    for kind, *args in scripted.pop(b, []):
                        log("  " + self.percept(kind, *args))
                    if self.position == goal:
                        break
                    remaining = plan.route[plan.route.index(b):]
                    forbidden = infer(self.kb, self.trailer).forbidden
                    if any((x, y) in forbidden for x, y in zip(remaining, remaining[1:])):
                        log(f"  Route ahead is now blocked -> re-planning from {b}")
                        plan = self.plan(goal)
                        log("  " + plan.explanation.replace("\n", "\n  "))
                        if not plan.ok:
                            log("  Cannot complete this trip.")
                            return False
                        replanned = True
                        break
            if not replanned:
                break
        log(f"Arrived at {self.position}.")
        return self.position == goal


Giving the user a way of interacting with the agent. For example by accepting text.

In [ ]:
%%writefile trailer/cli.py

from __future__ import annotations

import shlex
import sys

from .agent import TrailerAgent
from .kb import KnowledgeBase
from .rules import TrailerSpec, infer
from .search import format_tree, search

HELP = """Commands (use quotes or underscores for multi-word names, e.g. Mtito_Andei):
  places                        list places and facilities
  at <place>                    set the trailer's current position
  trailer <tonnes>              set gross weight (changes which roads are legal)
  distance <A> <B>              haversine / euclidean / manhattan / road distances
  route <A> <B> [strategy]      plan a route + fuel/rest stops   (bfs|ucs|greedy|astar)
  tree <A> <B> [strategy]       show the search tree
  why                           show roads the inference engine has forbidden, and why
  close <A> <B> / open <A> <B>  report a road closure / reopening
  ask <pred> [args...]          query the KB, '*' = wildcard, e.g. ask fuel_station *
  simulate <A> <B>              drive the plan step by step
  help | quit"""


def handle(agent: TrailerAgent, line: str) -> str:
    parts = shlex.split(line)
    if not parts:
        return ""
    cmd, args, kb = parts[0].lower(), parts[1:], agent.kb
    try:
        if cmd == "help":
            return HELP
        if cmd == "places":
            return "\n".join(f"  {p.name:<12} fuel={'Y' if p.fuel else '-'} rest={'Y' if p.rest else '-'}"
                             for p in kb.places.values())
        if cmd == "at":
            agent.position = kb.resolve(args[0])
            return f"Position: {agent.position}"
        if cmd == "trailer":
            agent.trailer.gross_tonnes = float(args[0])
            return f"Trailer gross weight: {agent.trailer.gross_tonnes:g} t"
        if cmd == "distance":
            return "\n".join(f"  {k:<13}{v}" for k, v in agent.distances(args[0], args[1]).items())
        if cmd == "route":
            plan = agent.plan(args[1], args[2] if len(args) > 2 else None, start=args[0])
            return plan.explanation + ("\n" + plan.trip.format() if plan.trip else "")
        if cmd == "tree":
            a, b = kb.resolve(args[0]), kb.resolve(args[1])
            return format_tree(search(kb, a, b, args[2] if len(args) > 2 else "astar",
                                      infer(kb, agent.trailer).forbidden))
        if cmd == "why":
            inf = infer(kb, agent.trailer)
            return "\n".join(f"  {a} -> {b}: {'; '.join(sorted(set(w)))}" for (a, b), w in sorted(inf.reasons.items())) \
                or "  Nothing is forbidden."
        if cmd in ("close", "open"):
            return agent.percept(cmd, args[0], args[1])
        if cmd == "ask":
            rows = kb.ask(*args)
            return "\n".join("  " + str(r) for r in rows) or "  (no matching facts)"
        if cmd == "simulate":
            agent.position = kb.resolve(args[0])
            out: list = []
            agent.drive(args[1], log=out.append)
            return "\n".join(out)
        return f"Unknown command {cmd!r}. Type 'help'."
    except (IndexError, ValueError):
        return "Bad arguments. Type 'help'."
    except KeyError as exc:
        return str(exc.args[0])


def demo() -> None:
    kb = KnowledgeBase.from_json()
    agent = TrailerAgent(kb, TrailerSpec(gross_tonnes=22))
    steps = [
        "distance Nairobi Kisumu",
        "tree Nairobi Kisumu astar",
        "route Nairobi Kisumu",
        "trailer 28",
        "why",
        "route Nairobi Kisumu",
        "route Mombasa Malaba",
        "ask fuel_station *",
    ]
    for s in steps:
        print(f"\n> {s}\n{handle(agent, s)}")
    print("\n> simulate with a closure ahead (22 t trailer)")
    a2 = TrailerAgent(KnowledgeBase.from_json(), TrailerSpec(gross_tonnes=22), position="Nairobi")
    a2.drive("Kisumu", {"Naivasha": [("close", "Nakuru", "Kericho")]})


def main(argv=None) -> None:
    argv = sys.argv[1:] if argv is None else argv
    if argv and argv[0] == "demo":
        return demo()
    agent = TrailerAgent(KnowledgeBase.from_json(), TrailerSpec())
    print("Self-driving trailer agent. Type 'help'.")
    while True:
        try:
            line = input("trailer> ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if line.lower() in ("quit", "exit"):
            break
        out = handle(agent, line)
        if out:
            print(out)


if __name__ == "__main__":
    main()


Demonstration of the Interaction:

In [ ]:
from trailer.agent import TrailerAgent
from trailer.cli import handle

agent = TrailerAgent(KnowledgeBase.from_json(), TrailerSpec(gross_tonnes=22), position='Nairobi')
for cmd in ['distance Nairobi Kisumu', 'route Nairobi Kisumu', 'trailer 28', 'why', 'route Nairobi Kisumu', 'route Mombasa Malaba']:
    print('>', cmd); print(handle(agent, cmd)); print()

Replanning of the Agent Mid-Trip on Road Closure:

In [ ]:
a = TrailerAgent(KnowledgeBase.from_json(), TrailerSpec(gross_tonnes=22), position='Nairobi')
a.drive('Kisumu', {'Naivasha': [('close', 'Nakuru', 'Kericho')]})

Type commands to query the agent yourself, for example 'route Nairobi Kisumu', 'trailer 28' ,
'close Nakuru Kericho', or 'ask fuel_station'.
Type 'quit' to stop.

In [ ]:
# Interactive: type commands, e.g. route Nairobi Kisumu   (type 'help' for all; 'quit' to stop)
agent = TrailerAgent(KnowledgeBase.from_json(), TrailerSpec())
while True:
    line = input('trailer> ').strip()
    if line.lower() in ('quit', 'exit'): break
    print(handle(agent, line))